# Inference

Depreciated, now moved over to running in google colab, and DEMs taken from DEFRA api instead of downloading and merging tifs.

In [ ]:
import rasterio as rio
import re
from rasterio.transform import from_bounds
from rasterio.crs import CRS
import numpy as np
import glob 
from pathlib import Path
import sys
import torch
from scipy.ndimage import label as ndlabel
from contextlib import ExitStack
from rasterio.merge import merge
import sys
sys.path.append('../../modelling/models')
sys.path.append('../../modelling/loaders')

from HillShadeUnet import UNet
from loader import get_test_loader, TestSCDDataset

In [ ]:
#CONFIG 
pred_dir = Path('../../../data/HillShade/BCEast2/out/preds')
tile_paths = glob.glob('../../../data/HillShade/BCEast2/out/tiles/*.npz')
TARGET_CRS = CRS.from_epsg(27700)

In [ ]:
model = UNet(in_channels=1, base_filters=32)
model.load_state_dict(torch.load('../../modelling/models/IRPHillShade.pt', map_location='cpu'))
test = TestSCDDataset(tile_paths)
test_loader = get_test_loader(test, batch_size=1)

In [ ]:
def parse_tile_bounds(path):
    stem = Path(path).stem
    nums = re.findall(r'\d+', stem)
    xmin, ymin, xmax, ymax = int(nums[-4]), int(nums[-3]), int(nums[-2]), int(nums[-1])
    return xmin, ymin, xmax, ymax

def count_objects(pred_mask, min_pixels=5):
    labelled, num_features = ndlabel(pred_mask)
    sizes = np.bincount(labelled.ravel())[1:]
    return int(np.sum(sizes >= min_pixels))

def save_pred_as_tiff(pred, bounds, out_path, crs, count):
    minx, miny, maxx, maxy = bounds
    h, w = pred.shape
    transform = from_bounds(minx, miny, maxx, maxy, w, h)
    with rio.open(
        str(out_path), 'w',
        driver='GTiff',
        height=h, width=w,
        count=1, dtype=np.uint8,
        crs=crs, transform=transform,
        compress='lzw',
        tags={'object_count': str(count)}
    ) as dst:
        dst.write((pred > 0).astype(np.uint8), 1)

def mosaic_predictions(pred_paths, out_path, crs):
    if not pred_paths:
        print("No predictions to mosaic.")
        return
    with ExitStack() as stack:
        srcs = [stack.enter_context(rio.open(p)) for p, _ in pred_paths]
        mosaic, mosaic_transform = merge(srcs, method='max', nodata=0, dtype=np.uint8)
    with rio.open(
        str(out_path), 'w',
        driver='GTiff',
        height=mosaic.shape[1], width=mosaic.shape[2],
        count=1, dtype=np.uint8,
        crs=crs, transform=mosaic_transform,
        compress='lzw',
    ) as dst:
        dst.write(mosaic[0], 1)
    print(f"Mosaic saved → {out_path}  ({mosaic.shape[2]}×{mosaic.shape[1]} px)")

def save_density_raster(tile_counts, pred_paths, out_path, crs):
    bounds_lookup = {p: b for p, b in pred_paths}
    first_bounds = next(iter(bounds_lookup.values()))
    tile_w = first_bounds[2] - first_bounds[0]
    tile_h = first_bounds[3] - first_bounds[1]

    all_bounds = list(bounds_lookup.values())
    global_xmin = min(b[0] for b in all_bounds)
    global_ymin = min(b[1] for b in all_bounds)
    global_xmax = max(b[2] for b in all_bounds)
    global_ymax = max(b[3] for b in all_bounds)

    cols = round((global_xmax - global_xmin) / tile_w)
    rows = round((global_ymax - global_ymin) / tile_h)
    grid = np.zeros((rows, cols), dtype=np.uint16)

    for tile_path, count in tile_counts.items():
        xmin, _, _, ymax = bounds_lookup[tile_path]
        col = round((xmin - global_xmin) / tile_w)
        row = round((global_ymax - ymax) / tile_h)
        grid[row, col] = count

    transform = from_bounds(global_xmin, global_ymin, global_xmax, global_ymax, cols, rows)
    with rio.open(
        str(out_path), 'w',
        driver='GTiff',
        height=rows, width=cols,
        count=1, dtype=np.uint16,
        crs=crs, transform=transform,
        compress='lzw', nodata=0,
    ) as dst:
        dst.write(grid, 1)
    print(f"Density raster saved → {out_path}  ({cols}×{rows} tiles)")



In [ ]:
model.eval()
pred_paths  = []
tile_counts = {}

with torch.no_grad():
    for i, (image, path) in enumerate(test_loader, start=1):

        if i % 10 == 0:
            print(f'Predicting tile {i} / {len(test_loader)}')

        logits = model(image)
        probs = torch.sigmoid(logits)
        pred = (probs > 0.85).float().squeeze().cpu().numpy()
            
        count = count_objects(pred)

        if count > 0:
            bounds = parse_tile_bounds(path[0])
            out_path = pred_dir / f"pred_{bounds[0]}_{bounds[1]}_{bounds[2]}_{bounds[3]}.tif"
            save_pred_as_tiff(pred, bounds, out_path, TARGET_CRS, count)
            pred_paths.append((out_path, bounds))
            tile_counts[out_path] = count

mosaic_predictions(pred_paths, pred_dir / "mosaic.tif",  TARGET_CRS)
save_density_raster(tile_counts, pred_paths, pred_dir / "density.tif", TARGET_CRS)